# Methanol Example Runner: Kinetic Batch-First Aspen Workflow

This notebook is the methanol-only example, regression, kinetic remediation, and 10k TPD tuning worksheet. Use `process_library_runner.ipynb` for the process-agnostic workflow for new user-defined processes.


## 1. Environment setup and imports

In [1]:
from __future__ import annotations

import importlib
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd()).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import aspen_automation.extractor as _aspen_extractor
import aspen_automation as _aspen_automation

importlib.reload(_aspen_extractor)
importlib.reload(_aspen_automation)

from aspen_automation import (
    calculate_synthesis_loop_diagnostics,
    analyze_process_spec_coherence,
    apply_process_spec_improvements,
    build_codex_improvement_markdown,
    build_codex_results_markdown,
    build_codex_spec_markdown,
    generate_inp,
    load_process_spec,
    load_result_artifact_tables,
    load_spec,
    run_aspen_batch,
    run_methanol_tuning_campaign,
    run_process_batch_first,
    scan_process_library,
    suggest_process_spec_improvements,
    validate_process_spec_file,
    write_process_spec_file,
)


## 1.5 Aspen Plus pre-flight check

In [2]:
from aspen_automation import check_aspen_running, AspenNotRunningError

if not check_aspen_running():
    raise AspenNotRunningError()

print('Aspen Plus is running. Ready to proceed.')

Aspen Plus is running. Ready to proceed.


## 2. Repository path resolution

In [3]:
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
PROCESS_LIBRARY_DIR = REPO_ROOT / "process_library"
PROCESS_RUNS_DIR = REPO_ROOT / "process_runs" / "batch_first_capsule"
GATE1_BATCH_ROOT = PROCESS_RUNS_DIR / "gate1_batch_translator"

print(f"Repository root: {REPO_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Process library: {PROCESS_LIBRARY_DIR}")
print(f"Batch-first process runs: {PROCESS_RUNS_DIR}")
print(f"Gate 1 batch translator workspace: {GATE1_BATCH_ROOT}")


Repository root: C:\Users\domingueza\ASPEN_PY
Notebook directory: C:\Users\domingueza\ASPEN_PY\notebooks
Process library: C:\Users\domingueza\ASPEN_PY\process_library
Batch-first process runs: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule
Gate 1 batch translator workspace: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator


## 3. Process library path configuration

In [4]:
LIBRARY_ROOT = PROCESS_LIBRARY_DIR
RUNS_ROOT = PROCESS_RUNS_DIR
VISIBLE = True
TIMEOUT_SECONDS = 1800
BATCH_TIMEOUT_SECONDS = 1800
REPORT_FORMAT = "html"
ENFORCE_ACCEPTANCE_TARGETS = False
REQUIRE_GATE1_SUCCESS = True
AUTO_APPLY_SUGGESTED_IMPROVEMENTS = False
ONLY_PROCESSES: set[str] | None = {"methanol"}

print("Notebook configuration")
print(f"- LIBRARY_ROOT={LIBRARY_ROOT}")
print(f"- RUNS_ROOT={RUNS_ROOT}")
print(f"- GATE1_BATCH_ROOT={GATE1_BATCH_ROOT}")
print(f"- VISIBLE={VISIBLE}")
print("- WORKFLOW=batch-first capsule: generated INP -> Aspen batch .bkp -> COM BKP load/extraction")
print(f"- TIMEOUT_SECONDS={TIMEOUT_SECONDS}")
print(f"- BATCH_TIMEOUT_SECONDS={BATCH_TIMEOUT_SECONDS}")
print(f"- REPORT_FORMAT={REPORT_FORMAT}")
print(f"- ENFORCE_ACCEPTANCE_TARGETS={ENFORCE_ACCEPTANCE_TARGETS}")
print(f"- REQUIRE_GATE1_SUCCESS={REQUIRE_GATE1_SUCCESS}")
print(f"- AUTO_APPLY_SUGGESTED_IMPROVEMENTS={AUTO_APPLY_SUGGESTED_IMPROVEMENTS}")
print(f"- ONLY_PROCESSES={ONLY_PROCESSES}")


Notebook configuration
- LIBRARY_ROOT=C:\Users\domingueza\ASPEN_PY\process_library
- RUNS_ROOT=C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule
- GATE1_BATCH_ROOT=C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator
- VISIBLE=True
- WORKFLOW=batch-first capsule: generated INP -> Aspen batch .bkp -> COM BKP load/extraction
- TIMEOUT_SECONDS=1800
- BATCH_TIMEOUT_SECONDS=1800
- REPORT_FORMAT=html
- ENFORCE_ACCEPTANCE_TARGETS=False
- REQUIRE_GATE1_SUCCESS=True
- AUTO_APPLY_SUGGESTED_IMPROVEMENTS=False
- ONLY_PROCESSES={'methanol'}


## 4. YAML schema / expected fields overview

Each `process.yaml` uses the existing plant specification schema already supported by `aspen_automation`.

Required top-level sections:

- `metadata`
- `components`
- `properties`
- `flowsheet`
- `streams`
- `blocks`

Optional sections already supported by the generator:

- `flowsheeting_options`
- `chemistry`
- `reaction_sets`
- `targets`


## 5. Discovery of all available process folders

In [5]:
scan = scan_process_library(LIBRARY_ROOT)

selected_processes = [
    process
    for process in scan.processes
    if ONLY_PROCESSES is None or process.name in ONLY_PROCESSES
]

process_rows = [
    {
        "process_name": process.name,
        "process_dir": str(process.process_dir),
        "spec_path": str(process.spec_path),
    }
    for process in selected_processes
]
issue_rows = [
    {
        "process_name": issue.process_name,
        "process_dir": str(issue.process_dir),
        "message": issue.message,
    }
    for issue in scan.issues
]

print(f"Discovered {len(process_rows)} runnable process(es) after filtering.")
display(pd.DataFrame(process_rows))

if issue_rows:
    print(f"Found {len(issue_rows)} discovery issue(s).")
    display(pd.DataFrame(issue_rows))


Discovered 1 runnable process(es) after filtering.


,process_name,process_dir,spec_path
0,methanol,C:\Users\domingueza\ASPEN_PY\process_library\m...,C:\Users\domingueza\ASPEN_PY\process_library\m...


## 6. Shared helper functions

In [6]:
def read_json_file(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def first_history_messages(history: dict | None, *, limit: int = 5) -> list[str]:
    if not isinstance(history, dict):
        return []
    messages: list[str] = []
    for message in history.get("messages", []):
        if not isinstance(message, dict):
            continue
        severity = str(message.get("severity", "")).strip().lower()
        if severity in {"terminal", "severe", "error"} or history.get("input_translation_failed"):
            text = str(message.get("message", "")).strip()
            if text:
                messages.append(text)
        if len(messages) >= limit:
            break
    return messages


def summarize_process_result(result) -> dict[str, str]:
    acceptance = result.details.get("acceptance") if result.details else None
    acceptance_value = ""
    if isinstance(acceptance, dict) and "passed" in acceptance:
        acceptance_value = str(acceptance["passed"])

    generated_files = ", ".join(result.generated_files[:5])
    if len(result.generated_files) > 5:
        generated_files += ", ..."

    return {
        "process_name": result.process_name,
        "status": result.status,
        "acceptance_passed": acceptance_value,
        "report_dir": str(result.report_dir) if result.report_dir else "",
        "generated_files": generated_files,
        "error": result.error or "",
    }


def display_process_banner(name: str) -> None:
    display(Markdown(f"## Process: `{name}`"))


def first_issue_message(report: dict | None, *, severity: str = "error") -> str:
    if not isinstance(report, dict):
        return ""
    for issue in report.get("issues", report.get("errors", [])):
        if issue.get("severity") == severity:
            return str(issue.get("message", ""))
    return ""


def parse_improvement_selection(response: str, suggestions: list[dict]) -> list[str]:
    auto_suggestions = [suggestion for suggestion in suggestions if suggestion.get("auto_applicable")]
    normalized = response.strip().lower()
    if normalized in {"", "n", "no"}:
        return []
    if normalized in {"y", "yes", "all"}:
        return [suggestion["id"] for suggestion in auto_suggestions]

    selected_ids: list[str] = []
    for token in normalized.split(","):
        token = token.strip()
        if not token.isdigit():
            continue
        index = int(token) - 1
        if 0 <= index < len(auto_suggestions):
            selected_ids.append(auto_suggestions[index]["id"])
    return selected_ids


def write_live_aspen_summary(result) -> tuple[Path | None, dict]:
    layout = getattr(result, "layout", None)
    if layout is None:
        return None, {}

    results_dir = Path(layout.results_dir)
    build_diagnostics = read_json_file(results_dir / "build_diagnostics.json")
    simulation_diagnostics = read_json_file(results_dir / "simulation_diagnostics.json")
    context_probe = read_json_file(results_dir / "context_probe.json")
    acceptance = read_json_file(results_dir / "acceptance.json")
    artifact_paths, artifact_tables = load_result_artifact_tables(result)

    batch_engine = build_diagnostics.get("batch_engine", {}) if isinstance(build_diagnostics, dict) else {}
    artifacts = batch_engine.get("artifacts", {}) if isinstance(batch_engine, dict) else {}
    csv_shapes = {
        name: {"rows": int(len(table)), "columns": [str(column) for column in table.columns]}
        for name, table in artifact_tables.items()
    }

    summary = {
        "process_name": result.process_name,
        "status": result.status,
        "succeeded": bool(result.succeeded),
        "error": result.error,
        "process_dir": str(result.process_dir),
        "spec_path": str(result.spec_path) if result.spec_path else None,
        "run_dir": str(layout.run_dir),
        "generated_inp_path": str(layout.generated_inp_path),
        "output_apw_path": str(layout.output_apw_path),
        "results_dir": str(layout.results_dir),
        "report_dir": str(result.report_dir) if result.report_dir else None,
        "context_probe_path": str(results_dir / "context_probe.json"),
        "build_diagnostics_path": str(results_dir / "build_diagnostics.json"),
        "simulation_diagnostics_path": str(results_dir / "simulation_diagnostics.json"),
        "acceptance_path": str(results_dir / "acceptance.json"),
        "history_path": artifacts.get(".his", {}).get("path"),
        "stdout_path": batch_engine.get("stdout_path"),
        "stderr_path": batch_engine.get("stderr_path"),
        "artifact_paths": {name: str(path) for name, path in artifact_paths.items()},
        "csv_shapes": csv_shapes,
        "model_quality_warnings": simulation_diagnostics.get("model_quality_warnings", []),
        "nrtl_binary_parameters_status": simulation_diagnostics.get("nrtl_binary_parameters_status"),
        "build": build_diagnostics,
        "simulation": simulation_diagnostics,
        "acceptance": acceptance,
        "context_probe": context_probe,
    }

    summary_path = Path(layout.run_dir) / "live_aspen_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    return summary_path, summary


## 7. YAML coherence analysis per discovered process

In [7]:
validation_reports: dict[str, dict] = {}
coherence_reports: dict[str, dict] = {}
process_specs: dict[str, dict] = {}
coherence_rows: list[dict[str, str]] = []

for process in selected_processes:
    display_process_banner(process.name)
    print(f"Process directory: {process.process_dir}")
    print(f"Spec path: {process.spec_path}")

    validation_report = validate_process_spec_file(process.spec_path)
    validation_reports[process.name] = validation_report

    if not validation_report.get("valid", False):
        display(Markdown(build_codex_spec_markdown(process.name, process.spec_path, validation_report, None)))
        coherence_rows.append(
            {
                "process_name": process.name,
                "validation_passed": "False",
                "coherence_passed": "False",
                "blocking_issue": first_issue_message({"issues": validation_report.get("errors", [])}) or "Validation failed",
            }
        )
        continue

    spec = load_process_spec(process.process_dir)
    process_specs[process.name] = spec
    coherence_report = analyze_process_spec_coherence(spec)
    coherence_reports[process.name] = coherence_report

    display(
        Markdown(
            build_codex_spec_markdown(
                process.name,
                process.spec_path,
                validation_report,
                coherence_report,
                spec=spec,
            )
        )
    )

    coherence_rows.append(
        {
            "process_name": process.name,
            "validation_passed": "True",
            "coherence_passed": str(coherence_report.get("passed", False)),
            "blocking_issue": first_issue_message(coherence_report) or "",
        }
    )

if coherence_rows:
    display(pd.DataFrame(coherence_rows))


## Process: `methanol`

Process directory: C:\Users\domingueza\ASPEN_PY\process_library\methanol
Spec path: C:\Users\domingueza\ASPEN_PY\process_library\methanol\process.yaml


### Codex Session: YAML Coherence `methanol`

- Spec path: `C:\Users\domingueza\ASPEN_PY\process_library\methanol\process.yaml`
- Property method: `NRTL`
- Block lineup: `MIX-FEED -> B-ATR -> B-COOL -> B-FLASH -> B-COMP -> MIX-LOOP -> B-SYN -> B-SEP -> SPLIT -> B-DIST`
- Structural validation: passed
- Coherence gate: passed

Warnings:
- `targets.purity`: High-purity target 0.998500 for stream 'MEOH-PRO' is delegated to a 'SEP' block, which is a coarse screening model for final purification. Suggestion: Keep this as a buildable screening model, or implement supported rigorous-column generation before moving the final purification step to RADFRAC.

- Execution gate: passed. The process can proceed to INP generation and Aspen execution.

,process_name,validation_passed,coherence_passed,blocking_issue
0,methanol,True,True,


## 8. Suggested YAML improvements per discovered process

In [8]:
improvement_plans: dict[str, list[dict]] = {}

for process in selected_processes:
    display_process_banner(process.name)

    validation_report = validation_reports.get(process.name)
    coherence_report = coherence_reports.get(process.name)
    spec = process_specs.get(process.name)

    if not isinstance(validation_report, dict) or not validation_report.get("valid", False):
        print("Skipping YAML improvement suggestions because structural validation did not pass.")
        continue

    if spec is None:
        spec = load_process_spec(process.process_dir)
        process_specs[process.name] = spec

    suggestions = suggest_process_spec_improvements(spec, coherence_report)
    improvement_plans[process.name] = suggestions
    display(Markdown(build_codex_improvement_markdown(process.name, suggestions)))

    auto_suggestions = [suggestion for suggestion in suggestions if suggestion.get("auto_applicable")]
    if not isinstance(coherence_report, dict) or coherence_report.get("passed", False):
        print("No blocking coherence issues remain, so no YAML edits are required before execution.")
        continue

    if not auto_suggestions:
        print("No automatic YAML improvements are available for the current coherence failures.")
        continue

    if not AUTO_APPLY_SUGGESTED_IMPROVEMENTS:
        print("AUTO_APPLY_SUGGESTED_IMPROVEMENTS=False; suggestions were displayed but no YAML changes were applied.")
        continue

    selected_ids = [str(suggestion["id"]) for suggestion in auto_suggestions if "id" in suggestion]
    if not selected_ids:
        print("No YAML changes were applied.")
        continue

    updated_spec, applied_suggestions = apply_process_spec_improvements(spec, suggestions, selected_ids=selected_ids)
    backup_path = write_process_spec_file(process.spec_path, updated_spec)
    print(f"Backup written to: {backup_path}")
    print("Applied YAML improvements:")
    for suggestion in applied_suggestions:
        print(f"- {suggestion['title']}")

    updated_validation_report = validate_process_spec_file(process.spec_path)
    validation_reports[process.name] = updated_validation_report

    if updated_validation_report.get("valid", False):
        updated_spec_loaded = load_process_spec(process.process_dir)
        process_specs[process.name] = updated_spec_loaded
        updated_coherence_report = analyze_process_spec_coherence(updated_spec_loaded)
    else:
        updated_spec_loaded = None
        updated_coherence_report = None

    coherence_reports[process.name] = updated_coherence_report
    display(
        Markdown(
            build_codex_spec_markdown(
                process.name,
                process.spec_path,
                updated_validation_report,
                updated_coherence_report,
                spec=updated_spec_loaded,
            )
        )
    )


## Process: `methanol`

### Suggested YAML Improvements: `methanol`

Additional manual review items:
- `Plan a supported RADFRAC upgrade`: Stream `MEOH-PRO` is produced by `B-DIST`, currently typed as `SEP`, which is a coarse screening model for the high-purity target. Expected effect: Would improve purification rigor after the generator supports column stages, feeds, condenser/reboiler settings, and product specifications.

No blocking coherence issues remain, so no YAML edits are required before execution.


## 9. Gate 1: Aspen batch translator

This gate compiles the generated INP with Aspen's batch engine and treats the `.his` file as the source of truth. A `.bkp` alone is not enough: the history status must be clean before Gate 2 runs.


In [9]:
gate1_rows: list[dict[str, object]] = []
gate1_batch_results: dict[str, object] = {}
GATE1_BATCH_ROOT.mkdir(parents=True, exist_ok=True)

for process in selected_processes:
    display_process_banner(process.name)
    print(f"Gate 1 process directory: {process.process_dir}")

    validation_report = validation_reports.get(process.name)
    coherence_report = coherence_reports.get(process.name)
    if not isinstance(validation_report, dict) or not validation_report.get("valid", False):
        print("Skipping Gate 1 because structural validation did not pass.")
        gate1_rows.append({"process_name": process.name, "gate1_status": "validation_failed", "ready_for_gate2": False})
        continue
    if not isinstance(coherence_report, dict) or not coherence_report.get("passed", False):
        print("Skipping Gate 1 because YAML coherence did not pass.")
        gate1_rows.append({"process_name": process.name, "gate1_status": "coherence_failed", "ready_for_gate2": False})
        continue

    gate_dir = GATE1_BATCH_ROOT / process.name
    gate_dir.mkdir(parents=True, exist_ok=True)
    spec = load_spec(str(process.spec_path))
    gate1_inp_path = gate_dir / f"{process.name}_generated.inp"
    generate_inp(spec, output_path=str(gate1_inp_path))

    batch_result = run_aspen_batch(
        gate1_inp_path,
        gate_dir / "batch",
        run_id=process.name,
        timeout_seconds=BATCH_TIMEOUT_SECONDS,
    )
    gate1_batch_results[process.name] = batch_result
    history = batch_result.history_diagnostics
    history_messages = first_history_messages(history)

    print(f"Gate 1 succeeded: {batch_result.succeeded}")
    print(f"History status: {history.get('status') if isinstance(history, dict) else 'unknown'}")
    print(f"Archive path: {batch_result.archive_path}")
    print(f"History path: {batch_result.artifacts.get('.his', {}).get('path')}")
    print(f"Stdout path: {batch_result.stdout_path}")
    print(f"Stderr path: {batch_result.stderr_path}")
    if history_messages:
        print("First blocking history messages:")
        for message in history_messages:
            print("- " + message.replace("\n", " | "))

    gate1_rows.append(
        {
            "process_name": process.name,
            "gate1_status": history.get("status") if isinstance(history, dict) else "unknown",
            "ready_for_gate2": bool(batch_result.succeeded),
            "archive_path": batch_result.archive_path or "",
            "history_path": batch_result.artifacts.get(".his", {}).get("path", ""),
            "stdout_path": batch_result.stdout_path,
            "stderr_path": batch_result.stderr_path,
            "blocking_messages": " | ".join(history_messages[:2]),
        }
    )

if gate1_rows:
    display(pd.DataFrame(gate1_rows))


## Process: `methanol`

Gate 1 process directory: C:\Users\domingueza\ASPEN_PY\process_library\methanol
Gate 1 succeeded: True
History status: converged
Archive path: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator\methanol\batch\methan07.bkp
History path: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator\methanol\batch\methan07.his
Stdout path: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator\methanol\batch\methan07.stdout.txt
Stderr path: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\gate1_batch_translator\methanol\batch\methan07.stderr.txt


,process_name,gate1_status,ready_for_gate2,archive_path,history_path,stdout_path,stderr_path,blocking_messages
0,methanol,converged,True,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,


## 10. Gate 2: Kinetic BKP load, COM extraction, and reports

This gate runs the full kinetic batch-first capsule path for each process: generated INP, Aspen batch `.bkp`, COM `InitFromArchive2`, convergence run, result extraction, and report generation. Acceptance targets are displayed but not enforced by default.


In [10]:
summary_rows: list[dict[str, str]] = []
process_results: list[object] = []
live_summary_rows: list[dict[str, object]] = []

for issue in scan.issues:
    display_process_banner(issue.process_name)
    print(f"Discovery error: {issue.message}")
    summary_rows.append(
        {
            "process_name": issue.process_name,
            "status": "discovery_failed",
            "acceptance_passed": "",
            "report_dir": "",
            "generated_files": "",
            "error": issue.message,
        }
    )

for process in selected_processes:
    display_process_banner(process.name)
    print(f"Process directory: {process.process_dir}")
    print(f"Spec path: {process.spec_path}")

    validation_report = validation_reports.get(process.name)
    coherence_report = coherence_reports.get(process.name)
    gate1_result = gate1_batch_results.get(process.name)

    if not isinstance(validation_report, dict) or not validation_report.get("valid", False):
        print("Skipping Gate 2 because structural validation did not pass.")
        summary_rows.append(
            {
                "process_name": process.name,
                "status": "validation_failed",
                "acceptance_passed": "",
                "report_dir": "",
                "generated_files": "",
                "error": first_issue_message({"issues": validation_report.get("errors", []) if isinstance(validation_report, dict) else []}) or "Validation failed",
            }
        )
        continue

    if not isinstance(coherence_report, dict) or not coherence_report.get("passed", False):
        print("Skipping Gate 2 because YAML coherence did not pass.")
        summary_rows.append(
            {
                "process_name": process.name,
                "status": "coherence_failed",
                "acceptance_passed": "",
                "report_dir": "",
                "generated_files": "",
                "error": first_issue_message(coherence_report) or "Coherence analysis failed",
            }
        )
        continue

    if REQUIRE_GATE1_SUCCESS and (gate1_result is None or not gate1_result.succeeded):
        print("Skipping Gate 2 because Gate 1 batch translation did not produce a clean archive.")
        summary_rows.append(
            {
                "process_name": process.name,
                "status": "gate1_failed",
                "acceptance_passed": "",
                "report_dir": "",
                "generated_files": "",
                "error": "Gate 1 batch translator did not pass",
            }
        )
        continue

    result = run_process_batch_first(
        process.process_dir,
        RUNS_ROOT,
        visible=VISIBLE,
        enforce_acceptance_targets=ENFORCE_ACCEPTANCE_TARGETS,
        timeout_seconds=TIMEOUT_SECONDS,
        batch_timeout_seconds=BATCH_TIMEOUT_SECONDS,
        report_format=REPORT_FORMAT,
    )

    print(f"Status: {result.status}")
    if result.layout:
        print(f"Run directory: {result.layout.run_dir}")
        print(f"Results directory: {result.layout.results_dir}")
        print(f"Context probe: {result.layout.results_dir / 'context_probe.json'}")
    if result.report_dir:
        print(f"Report directory: {result.report_dir}")
    if result.generated_files:
        print("Generated files:")
        for generated_file in result.generated_files:
            print(f"- {generated_file}")
    if result.error:
        print(f"Error: {result.error}")

    build_diagnostics = result.details.get("build_diagnostics", {}) if result.details else {}
    batch_engine = build_diagnostics.get("batch_engine", {}) if isinstance(build_diagnostics, dict) else {}
    history = build_diagnostics.get("history_diagnostics", {}) if isinstance(build_diagnostics, dict) else {}
    history_messages = first_history_messages(history)
    if batch_engine:
        print(f"Batch history: {batch_engine.get('artifacts', {}).get('.his', {}).get('path')}")
        print(f"Batch stdout: {batch_engine.get('stdout_path')}")
        print(f"Batch stderr: {batch_engine.get('stderr_path')}")
    if history_messages:
        print("First blocking history messages:")
        for message in history_messages:
            print("- " + message.replace("\n", " | "))

    acceptance = result.details.get("acceptance") if result.details else None
    if isinstance(acceptance, dict):
        print(f"Acceptance passed: {acceptance.get('passed')} (enforced={ENFORCE_ACCEPTANCE_TARGETS})")

    summary_path, live_summary = write_live_aspen_summary(result)
    if summary_path is not None:
        print(f"Live Aspen summary: {summary_path}")
        live_summary_rows.append(
            {
                "process_name": result.process_name,
                "summary_path": str(summary_path),
                "history_path": live_summary.get("history_path") or "",
                "context_probe_path": live_summary.get("context_probe_path") or "",
                "status": live_summary.get("status"),
            }
        )

    process_results.append(result)
    summary_rows.append(summarize_process_result(result))

if live_summary_rows:
    display(pd.DataFrame(live_summary_rows))


## Process: `methanol`

Process directory: C:\Users\domingueza\ASPEN_PY\process_library\methanol
Spec path: C:\Users\domingueza\ASPEN_PY\process_library\methanol\process.yaml
[15:01:47] INFO: Starting simulation run...
[15:01:50] INFO: Found convergence status at \Data\Convergence\Batch-Options\Output\PER_ERROR: 0.0 -> converged
[15:01:50] INFO: Simulation finished with status: converged in 2.01s
[15:01:53] INFO: Cleaning up session... Closing and quitting Aspen.
Convergence status: converged
KPI values:
- production_rate_tpd: 10048.293984000002
- product_stream: MEOH-PRO
- product_total_tpd: 10048.293984000002
- product_component: CH3OH
- product_component_tpd: 10047.065911654157
- methanol_tpd: 10047.065911654157
- purity_fraction: 0.999877783
- energy_consumption_mw: 1776.6457536208
- energy_unit_basis: {'raw_unit': 'CAL/SEC', 'converted_unit': 'MW', 'conversion': '1 CAL/SEC = 0.004184 kW = 0.000004184 MW'}
- yield_fraction: None
- convergence_status: converged
- synthesis_loop: {'block': 'B-SYN', 'availab

,process_name,summary_path,history_path,context_probe_path,status
0,methanol,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,succeeded


## 11. Gate 2 diagnostics and evidence bundle

Review the evidence generated by the capsule run. These paths are the bundle to use when debugging AppsAnywhere/Cloudpaging package context or escalating missing Aspen dependencies to IT.


In [11]:
diagnostic_rows: list[dict[str, object]] = []

for result in process_results:
    layout = getattr(result, "layout", None)
    if layout is None:
        continue

    results_dir = Path(layout.results_dir)
    build_diagnostics = read_json_file(results_dir / "build_diagnostics.json")
    simulation_diagnostics = read_json_file(results_dir / "simulation_diagnostics.json")
    context_probe = read_json_file(results_dir / "context_probe.json")
    acceptance = read_json_file(results_dir / "acceptance.json")
    summary_path = Path(layout.run_dir) / "live_aspen_summary.json"
    live_summary = read_json_file(summary_path)

    package_markers = context_probe.get("package_markers", {}) if isinstance(context_probe, dict) else {}
    localization = context_probe.get("localization_assembly", {}) if isinstance(context_probe, dict) else {}
    batch_engine = build_diagnostics.get("batch_engine", {}) if isinstance(build_diagnostics, dict) else {}
    artifacts = batch_engine.get("artifacts", {}) if isinstance(batch_engine, dict) else {}

    diagnostic_rows.append(
        {
            "process_name": result.process_name,
            "status": result.status,
            "run_status": simulation_diagnostics.get("run_status"),
            "results_status": simulation_diagnostics.get("results_status"),
            "nrtl_binary_parameters_status": simulation_diagnostics.get("nrtl_binary_parameters_status"),
            "model_quality_warning_count": len(simulation_diagnostics.get("model_quality_warnings", [])),
            "acceptance_passed": acceptance.get("passed"),
            "appanywhere_markers": package_markers.get("suspected_appanywhere_virtualization"),
            "localization_visible": localization.get("visible"),
            "history_path": artifacts.get(".his", {}).get("path"),
            "stdout_path": batch_engine.get("stdout_path"),
            "stderr_path": batch_engine.get("stderr_path"),
            "context_probe.json": str(results_dir / "context_probe.json"),
            "build_diagnostics.json": str(results_dir / "build_diagnostics.json"),
            "simulation_diagnostics.json": str(results_dir / "simulation_diagnostics.json"),
            "live_aspen_summary.json": str(summary_path) if summary_path.is_file() else "",
        }
    )

    display_process_banner(result.process_name)
    print(f"live_aspen_summary.json: {summary_path}")
    print(f"context_probe.json: {results_dir / 'context_probe.json'}")
    print(f"build_diagnostics.json: {results_dir / 'build_diagnostics.json'}")
    print(f"simulation_diagnostics.json: {results_dir / 'simulation_diagnostics.json'}")
    print(f"acceptance.json: {results_dir / 'acceptance.json'}")
    if live_summary:
        print(f"History: {live_summary.get('history_path')}")
        print(f"Stdout: {live_summary.get('stdout_path')}")
        print(f"Stderr: {live_summary.get('stderr_path')}")

if diagnostic_rows:
    display(pd.DataFrame(diagnostic_rows))
else:
    print("No Gate 2 diagnostic artifacts are available yet.")


## Process: `methanol`

live_aspen_summary.json: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\live_aspen_summary.json
context_probe.json: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\results\context_probe.json
build_diagnostics.json: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\results\build_diagnostics.json
simulation_diagnostics.json: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\results\simulation_diagnostics.json
acceptance.json: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\results\acceptance.json
History: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\session\batch\methanol.his
Stdout: C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\session\batch\methanol.stdout.txt


,process_name,status,run_status,results_status,nrtl_binary_parameters_status,model_quality_warning_count,acceptance_passed,appanywhere_markers,localization_visible,history_path,stdout_path,stderr_path,context_probe.json,build_diagnostics.json,simulation_diagnostics.json,live_aspen_summary.json
0,methanol,succeeded,converged,readable,accepted_by_aspen,0,True,True,False,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,C:\Users\domingueza\ASPEN_PY\process_runs\batc...


## 12. Codex session analysis per discovered process

This readout uses CSV artifacts as the source of truth and highlights B-SYN selectivity diagnostics, recycle composition diagnostics, low methanol formation, and KPI product-stream selection issues.


In [12]:
if not process_results:
    print("No process runs available for Codex session analysis.")
else:
    for result in process_results:
        display_process_banner(result.process_name)

        if not result.succeeded:
            print(f"Skipping Codex session analysis because status={result.status!r}.")
            continue

        artifact_paths, artifact_tables = load_result_artifact_tables(result)
        acceptance = result.details.get("acceptance") if result.details else None
        display(
            Markdown(
                build_codex_results_markdown(
                    result.process_name,
                    artifact_paths,
                    artifact_tables,
                    acceptance=acceptance,
                )
            )
        )


## Process: `methanol`

### Codex Session Analysis: `methanol`

Use the CSV artifacts below as the source of truth for interpretation:
- `streams.csv`: `C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\reports\run_2026-05-19_15-01-54\streams.csv`
- `blocks.csv`: `C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\reports\run_2026-05-19_15-01-54\blocks.csv`
- `material_balance.csv`: `C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\reports\run_2026-05-19_15-01-54\material_balance.csv`
- `energy_balance.csv`: `C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\reports\run_2026-05-19_15-01-54\energy_balance.csv`

CSV-based readout:
- `streams.csv`: 17 stream row(s) loaded.
- Largest mass-flow streams: R-OUT (3,179,529.0), R-IN (3,179,529.0), GAS-PURG (2,667,149.2).
- Strongest methanol-carrying stream: MEOH-PRO with `CH3OH_mass_frac`=1.
- B-SYN selectivity diagnostics: CO conversion 13.804%, CO2 conversion 23.386%, H2 consumption 17.724%, methanol change 13,309.1 kmol/hr, methane change 5.23e-06 kmol/hr.
- Recycle feed diagnostics: `R-IN` stoichiometric number 1.738, CH4 mole fraction 0.0345, CO2 mole fraction 0.0431.
- Low methanol diagnosis: prioritize B-SYN selectivity, recycle buildup, and KPI product-stream selection before nameplate tuning.
- `blocks.csv`: 10 block row(s) loaded.
- Largest absolute block duties: B-COOL (-1,077.7 MW), B-SEP (-607.632 MW), B-SYN (59.223 MW).
- Duty unit basis: raw Aspen values are CAL/SEC; summaries use corrected `duty_mw`.
- Net compressor/pump work from `blocks.csv`: 0.309 MW.
- `material_balance.csv`: 8 component row(s) loaded.
- Largest feed/product closure gaps: O2 (-100.000%), CH4 (-94.997%), H2O (11.732%).
- `energy_balance.csv`: 11 row(s) loaded.
- Energy summary: net duty -1,593.9 MW, heating 91.357 MW, cooling -1,685.3 MW.
- Acceptance status from `acceptance.json`: True.
- Diagnostic focus: the feed/product cut is not materially closed; the recycle feed stoichiometric number is below the methanol target range; the net heat duty magnitude is extreme.

## 13. Kinetic remediation and tuning worksheet

Use this after Gate 2 produces readable results. The first objective is stable nonzero methanol with credible B-SYN selectivity diagnostics and recycle composition diagnostics; 10k TPD remains displayed but not enforced by default.


In [13]:
PURGE_SWEEP_FRACTIONS = [0.05, 0.10, 0.15, 0.20, 0.30]
ATR_TUNING_FACTORS = [
    {"steam_factor": 1.00, "oxygen_factor": 1.00},
    {"steam_factor": 1.10, "oxygen_factor": 0.95},
    {"steam_factor": 1.20, "oxygen_factor": 0.90},
    {"steam_factor": 1.30, "oxygen_factor": 0.85},
]

kinetic_rows: list[dict[str, object]] = []
tuning_rows: list[dict[str, object]] = []

for result in process_results:
    display_process_banner(result.process_name)
    layout = getattr(result, "layout", None)
    if layout is None:
        print("No run layout available for kinetic remediation diagnostics.")
        continue

    kpis = read_json_file(Path(layout.results_dir) / "kpis.json")
    synthesis = kpis.get("synthesis_loop", {}) if isinstance(kpis, dict) else {}
    kinetic_rows.append(
        {
            "process_name": result.process_name,
            "product_stream": kpis.get("product_stream"),
            "product_total_tpd": kpis.get("product_total_tpd"),
            "methanol_tpd": kpis.get("methanol_tpd"),
            "product_component_tpd": kpis.get("product_component_tpd"),
            "co_conversion_fraction": synthesis.get("co_conversion_fraction"),
            "co2_conversion_fraction": synthesis.get("co2_conversion_fraction"),
            "h2_consumption_fraction": synthesis.get("h2_consumption_fraction"),
            "methanol_formation_kmol_hr": synthesis.get("methanol_formation_kmol_hr"),
            "methane_change_kmol_hr": synthesis.get("methane_change_kmol_hr"),
            "rin_stoichiometric_number": synthesis.get("inlet_stoichiometric_number"),
            "rin_ch4_mole_frac": synthesis.get("inlet_ch4_mole_frac"),
            "rin_co2_mole_frac": synthesis.get("inlet_co2_mole_frac"),
        }
    )

    print("B-SYN selectivity diagnostics:")
    print(json.dumps(synthesis, indent=2, default=str))
    print("Recycle composition diagnostics target: R-IN SN 1.8-2.2 and avoid CH4/CO2-dominated recycle.")

    for purge_fraction in PURGE_SWEEP_FRACTIONS:
        recycle_fraction = 1.0 - purge_fraction
        tuning_rows.append(
            {
                "process_name": result.process_name,
                "tuning_stage": "purge_fraction_sweep",
                "purge_fraction": purge_fraction,
                "recycle_fraction": recycle_fraction,
                "steam_factor": 1.0,
                "oxygen_factor": 1.0,
                "run_when": "after nonzero methanol is confirmed",
            }
        )

    for factors in ATR_TUNING_FACTORS:
        tuning_rows.append(
            {
                "process_name": result.process_name,
                "tuning_stage": "atr_feed_ratio_sweep",
                "purge_fraction": None,
                "recycle_fraction": None,
                "steam_factor": factors["steam_factor"],
                "oxygen_factor": factors["oxygen_factor"],
                "run_when": "after recycle is not CH4/CO2 dominated",
            }
        )

if kinetic_rows:
    display(pd.DataFrame(kinetic_rows))
if tuning_rows:
    display(pd.DataFrame(tuning_rows))


## Process: `methanol`

B-SYN selectivity diagnostics:
{
  "block": "B-SYN",
  "available": true,
  "inlet_stream": "R-IN",
  "outlet_stream": "R-OUT",
  "co_conversion_fraction": 0.13804480749132025,
  "co2_conversion_fraction": 0.23385964808028017,
  "h2_consumption_fraction": 0.17723622993905497,
  "methanol_formation_kmol_hr": 13309.056983321178,
  "methane_change_kmol_hr": 5.228661393630318e-06,
  "inlet_stoichiometric_number": 1.737871392051257,
  "inlet_ch4_mole_frac": 0.0344768088,
  "inlet_co2_mole_frac": 0.0430812806,
  "outlet_ch3oh_mole_frac": 0.05869416
}
Recycle composition diagnostics target: R-IN SN 1.8-2.2 and avoid CH4/CO2-dominated recycle.


,process_name,product_stream,product_total_tpd,methanol_tpd,product_component_tpd,co_conversion_fraction,co2_conversion_fraction,h2_consumption_fraction,methanol_formation_kmol_hr,methane_change_kmol_hr,rin_stoichiometric_number,rin_ch4_mole_frac,rin_co2_mole_frac
0,methanol,MEOH-PRO,10048.293984,10047.065912,10047.065912,0.138045,0.23386,0.177236,13309.056983,0.000005,1.737871,0.034477,0.043081


,process_name,tuning_stage,purge_fraction,recycle_fraction,steam_factor,oxygen_factor,run_when
0,methanol,purge_fraction_sweep,0.05,0.95,1.0,1.00,after nonzero methanol is confirmed
1,methanol,purge_fraction_sweep,0.10,0.90,1.0,1.00,after nonzero methanol is confirmed
2,methanol,purge_fraction_sweep,0.15,0.85,1.0,1.00,after nonzero methanol is confirmed
3,methanol,purge_fraction_sweep,0.20,0.80,1.0,1.00,after nonzero methanol is confirmed
4,methanol,purge_fraction_sweep,0.30,0.70,1.0,1.00,after nonzero methanol is confirmed
5,methanol,atr_feed_ratio_sweep,NaN,NaN,1.0,1.00,after recycle is not CH4/CO2 dominated
6,methanol,atr_feed_ratio_sweep,NaN,NaN,1.1,0.95,after recycle is not CH4/CO2 dominated
7,methanol,atr_feed_ratio_sweep,NaN,NaN,1.2,0.90,after recycle is not CH4/CO2 dominated
8,methanol,atr_feed_ratio_sweep,NaN,NaN,1.3,0.85,after recycle is not CH4/CO2 dominated


## 13.5 Live methanol production tuning campaign

Run this opt-in campaign after Gate 2 has produced readable results. It creates variant specs, runs batch-first Aspen simulations, writes `tuning_campaign_summary.csv`, `tuning_campaign_summary.json`, and `best_process.yaml`, and promotes the winner only if the moderate uplift stop rule is met.


In [14]:
RUN_METHANOL_TUNING_CAMPAIGN = False
METHANOL_TUNING_MAX_CASES = 40
METHANOL_TUNING_PROMOTE = True
METHANOL_TUNING_MIN_R_OUT_CH3OH = 0.01
METHANOL_TUNING_MIN_PRODUCT_CH3OH_TPD = 500.0

tuning_campaign_result = None
if RUN_METHANOL_TUNING_CAMPAIGN:
    tuning_campaign_result = run_methanol_tuning_campaign(
        PROCESS_LIBRARY_DIR / "methanol",
        PROCESS_RUNS_DIR,
        visible=VISIBLE,
        timeout_seconds=TIMEOUT_SECONDS,
        batch_timeout_seconds=BATCH_TIMEOUT_SECONDS,
        report_format=REPORT_FORMAT,
        max_cases=METHANOL_TUNING_MAX_CASES,
        promote=METHANOL_TUNING_PROMOTE,
        min_r_out_ch3oh_mole_frac=METHANOL_TUNING_MIN_R_OUT_CH3OH,
        min_product_ch3oh_tpd=METHANOL_TUNING_MIN_PRODUCT_CH3OH_TPD,
    )
    print(f"Tuning campaign: {tuning_campaign_result.campaign_dir}")
    print(f"tuning_campaign_summary.csv: {tuning_campaign_result.summary_csv_path}")
    print(f"tuning_campaign_summary.json: {tuning_campaign_result.summary_json_path}")
    print(f"best_process.yaml: {tuning_campaign_result.best_process_path}")
    print(f"promoted: {tuning_campaign_result.promoted}")
    if tuning_campaign_result.summary_csv_path.is_file():
        display(pd.read_csv(tuning_campaign_result.summary_csv_path))
else:
    print("Set RUN_METHANOL_TUNING_CAMPAIGN = True to run the 20-40 case live Aspen tuning campaign.")


Set RUN_METHANOL_TUNING_CAMPAIGN = True to run the 20-40 case live Aspen tuning campaign.


## 14. Output summary and validation


In [15]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if summary_df.empty:
    print("No processes were executed.")
else:
    print("Completed process library run.")
    print(summary_df[["process_name", "status", "acceptance_passed", "report_dir"]].to_string(index=False))


,process_name,status,acceptance_passed,report_dir,generated_files,error
0,methanol,succeeded,True,C:\Users\domingueza\ASPEN_PY\process_runs\batc...,"methanol_generated.inp, methanol_output.ads, m...",


Completed process library run.
process_name    status acceptance_passed                                                                                                                     report_dir
    methanol succeeded              True C:\Users\domingueza\ASPEN_PY\process_runs\batch_first_capsule\methanol\run_2026-05-19_15-01-30\reports\run_2026-05-19_15-01-54
